## 시도 1

### 드라이브 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/NoonGil'
os.makedirs(DRIVE_BASE, exist_ok=True)
print("드라이브 마운트 완료")

Mounted at /content/drive
드라이브 마운트 완료


### YOLOv12 설치

In [ ]:
import os
os.chdir('/content')

!git clone https://github.com/sunsmarterjie/yolov12.git
os.chdir('/content/yolov12')

!pip install -r requirements.txt -q
!pip install -e . -q

print("YOLOv12 설치 완료")

Cloning into 'yolov12'...
remote: Enumerating objects: 1173, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1173 (delta 0), reused 0 (delta 0), pack-reused 1172 (from 2)
Receiving objects: 100% (1173/1173), 1.95 MiB | 13.70 MiB/s, done.
Resolving deltas: 100% (531/531), done.
ERROR: flash_attn-2.7.3+cu11torch2.2cxx11abiFALSE-cp311-cp311-linux_x86_64.whl is not a supported wheel on this platform.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
YOLOv12 설치 완료


# v7~v9 학습 시도

## 추가 데이터셋 다운로드

In [ ]:
!pip install roboflow -q
from roboflow import Roboflow

API_KEY = 'vgvnQD3ktPGMXOXsiCG7'  # 기존 API key 유지
rf = Roboflow(api_key=API_KEY)
ws = 's-workspace-6sd3n'

# 기존 NoonGil 데이터셋 (v4)
print('=== NoonGil v4 다운로드 ===')
noongil = rf.workspace(ws).project('noongil-barrierfree-ai').version(4).download(
    'yolov8', location='/content/datasets/noongil')

# 기존 pothole 데이터셋
print('=== pothole 다운로드 ===')
pothole = rf.workspace(ws).project('pothole-vhmow-jp2rw').version(3).download(
    'yolov8', location='/content/datasets/pothole')

# 추가: step 데이터셋 (273장)
print('=== step 추가 데이터셋 다운로드 (273장) ===')
step_new = rf.workspace(ws).project('step-wfcvl-n5kgc').version(1).download(
    'yolov8', location='/content/datasets/step_new')

print('\n전체 다운로드 완료')

=== NoonGil v4 다운로드 ===
loading Roboflow workspace...
loading Roboflow project...
=== pothole 다운로드 ===
loading Roboflow workspace...
loading Roboflow project...
=== step 추가 데이터셋 다운로드 (273장) ===
loading Roboflow workspace...
loading Roboflow project...

전체 다운로드 완료


## 클래스 정의 & 병합 함수

In [ ]:
import shutil, random
from pathlib import Path

CLASS_NAMES = [
    'bench',            # 0
    'bicycle',          # 1
    'bollard',          # 2
    'clothing_bin',     # 3
    'cone',             # 4
    'electric_scooter', # 5
    'fire_hydrant',     # 6
    'motorcycle',       # 7
    'pavement_damage',  # 8
    'ramp',             # 9
    'step',             # 10
    'street_light',     # 11
    'trash',            # 12
    'tree'              # 13
]

# 클래스 매핑 정의
PUBLIC_CLASS_MAP = {
    'ramp':       {'0': '9',  '1': '10'},  # ramp→9, stairs→10
    'pothole':    {'0': '8'},               # pothole→pavement_damage(8)
    'step_new':   {'0': '10'},              # step→10
    'motorcycle': {'0': '7'},               # motorcycle→7
}

def remap_label(src_label_path, dst_label_path, class_map):
    with open(src_label_path, 'r') as f:
        lines = f.readlines()
    remapped = []
    for line in lines:
        parts = line.strip().split()
        if not parts:
            continue
        old_id = parts[0]
        new_id = class_map.get(old_id)
        if new_id is None:
            print(f'  ⚠️ 매핑 없는 클래스 ID: {old_id} → 스킵')
            continue
        remapped.append(f"{new_id} {' '.join(parts[1:])}\n")
    if remapped:
        with open(dst_label_path, 'w') as f:
            f.writelines(remapped)

def copy_split(src_base, dst_base, split, class_map=None, prefix=''):
    src_img = Path(src_base) / split / 'images'
    src_lbl = Path(src_base) / split / 'labels'
    dst_img = Path(dst_base) / split / 'images'
    dst_lbl = Path(dst_base) / split / 'labels'

    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    if not src_img.exists():
        print(f'  [{split}] 폴더 없음, 스킵')
        return 0

    images = list(src_img.glob('*.jpg')) + \
             list(src_img.glob('*.png')) + \
             list(src_img.glob('*.jpeg'))
    count = 0
    for img_path in images:
        stem = img_path.stem
        new_stem = f'{prefix}_{stem}'
        shutil.copy(img_path, dst_img / f'{new_stem}{img_path.suffix}')
        lbl_path = src_lbl / f'{stem}.txt'
        dst_lbl_path = dst_lbl / f'{new_stem}.txt'
        if lbl_path.exists():
            if class_map:
                remap_label(lbl_path, dst_lbl_path, class_map)
            else:
                shutil.copy(lbl_path, dst_lbl_path)
        count += 1
    print(f'  [{split}] {count}장 완료')
    return count

print('함수 정의 완료')

함수 정의 완료


## 데이터셋 병합

In [ ]:
MERGED = '/content/datasets/merged'

if os.path.exists(MERGED):
    shutil.rmtree(MERGED)
    print('기존 merged 폴더 삭제 완료')

# 기존 NoonGil 직접 수집 데이터
print('=== NoonGil (직접 수집) ===')
for split in ['train', 'valid', 'test']:
    copy_split('/content/datasets/noongil', MERGED, split,
               class_map=None, prefix='noongil')

# 기존 pothole 데이터
print('\n=== pothole (pothole→8) ===')
for split in ['train', 'valid', 'test']:
    copy_split('/content/datasets/pothole', MERGED, split,
               class_map=PUBLIC_CLASS_MAP['pothole'], prefix='pothole')

# 추가: step 데이터셋 (절반만 사용)
print('\n=== step 추가 데이터셋 (50% 샘플링, step→10) ===')
for split in ['train', 'valid', 'test']:
    src_img = Path(f'/content/datasets/step_new/{split}/images')
    src_lbl = Path(f'/content/datasets/step_new/{split}/labels')
    dst_img = Path(f'{MERGED}/{split}/images')
    dst_lbl = Path(f'{MERGED}/{split}/labels')

    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    if src_img.exists():
        images = list(src_img.glob('*.jpg')) + \
                 list(src_img.glob('*.png')) + \
                 list(src_img.glob('*.jpeg'))
        # 절반만 랜덤 샘플링
        half = random.sample(images, len(images) // 2)
        for img_path in half:
            stem = img_path.stem
            new_stem = f'step_new_{stem}'
            shutil.copy(img_path, dst_img / f'{new_stem}{img_path.suffix}')
            lbl_path = src_lbl / f'{stem}.txt'
            dst_lbl_path = dst_lbl / f'{new_stem}.txt'
            if lbl_path.exists():
                remap_label(lbl_path, dst_lbl_path, PUBLIC_CLASS_MAP['step_new'])
        print(f'  [{split}] {len(half)}장 완료 (전체 {len(images)}장 중)')

# 최종 장수 확인
print('\n=== 병합 결과 ===')
for split in ['train', 'valid', 'test']:
    imgs = list(Path(f'{MERGED}/{split}/images').glob('*'))
    lbls = list(Path(f'{MERGED}/{split}/labels').glob('*.txt'))
    print(f'{split}: 이미지 {len(imgs)}장 / 라벨 {len(lbls)}개')

기존 merged 폴더 삭제 완료
=== NoonGil (직접 수집) ===
  [train] 780장 완료
  [valid] 74장 완료
  [test] 38장 완료

=== pothole (pothole→8) ===
  [train] 288장 완료
  [valid] 84장 완료
  [test] 40장 완료

=== step 추가 데이터셋 (50% 샘플링, step→10) ===
  [train] 106장 완료 (전체 212장 중)
  [valid] 20장 완료 (전체 41장 중)
  [test] 10장 완료 (전체 20장 중)

=== 병합 결과 ===
train: 이미지 1174장 / 라벨 1075개
valid: 이미지 178장 / 라벨 148개
test: 이미지 88장 / 라벨 73개


## data.yaml 생성

In [ ]:
import yaml

data_yaml = {
    'path': MERGED,
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc': 14,
    'names': [
        'bench', 'bicycle', 'bollard', 'clothing_bin', 'cone',
        'electric_scooter', 'fire_hydrant', 'motorcycle',
        'pavement_damage', 'ramp', 'step', 'street_light', 'trash', 'tree'
    ]
}

yaml_path = f'{MERGED}/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, allow_unicode=True, default_flow_style=False)

shutil.copy(yaml_path, f'{DRIVE_BASE}/v10_finetune_data.yaml')
print('data.yaml 생성 완료')
!cat {yaml_path}

data.yaml 생성 완료
names:
- bench
- bicycle
- bollard
- clothing_bin
- cone
- electric_scooter
- fire_hydrant
- motorcycle
- pavement_damage
- ramp
- step
- street_light
- trash
- tree
nc: 14
path: /content/datasets/merged
test: test/images
train: train/images
val: valid/images


## fine-tuning 학습

In [ ]:
import os
os.chdir('/content/yolov12')

BASE_WEIGHTS = f'{DRIVE_BASE}/v12_no_curb_ep100_best.pt'

# 가중치 파일 존재 확인
if not os.path.exists(BASE_WEIGHTS):
    print(f'❌ 기존 가중치 없음: {BASE_WEIGHTS}')
    print('Google Drive의 NoonGil 폴더에 v12_no_curb_ep100_best.pt 업로드 후 재실행')
else:
    print(f'✅ 기존 가중치 확인: {BASE_WEIGHTS}')
    !yolo detect train \
      data="/content/datasets/merged/data.yaml" \
      model={BASE_WEIGHTS} \
      epochs=50 \
      imgsz=640 \
      batch=16 \
      workers=2 \
      lr0=0.001 \
      freeze=10 \
      project={DRIVE_BASE}/runs \
      name=v12_no_curb_ep100_finetune_v10 \
      exist_ok=True

✅ 기존 가중치 확인: /content/drive/MyDrive/NoonGil/v12_no_curb_ep100_best.pt
FlashAttention is not available on this device. Using scaled_dot_product_attention instead.
New https://pypi.org/project/ultralytics/8.4.70 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=/content/drive/MyDrive/NoonGil/v12_no_curb_ep100_best.pt, data=/content/datasets/merged/data.yaml, epochs=50, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=2, project=/content/drive/MyDrive/NoonGil/runs, name=v12_no_curb_ep100_finetune_v10, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=10, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save

## 결과 확인 & 드라이브 저장

In [ ]:
import pandas as pd

results_dir = f'{DRIVE_BASE}/runs/v12_no_curb_ep100_finetune_v10'
best_pt = f'{results_dir}/weights/best.pt'

if os.path.exists(best_pt):
    shutil.copy(best_pt, f'{DRIVE_BASE}/v12_no_curb_finetune_v10_best.pt')
    print('✅ best.pt 드라이브 저장 완료')
    print(f'저장 위치: {DRIVE_BASE}/v12_no_curb_finetune_v10_best.pt')
else:
    print('❌ 학습 완료 후 실행해줘')

csv_path = f'{results_dir}/results.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print('\n=== 최근 10 epoch 결과 ===')
    print(df[['epoch', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']].tail(10))

✅ best.pt 드라이브 저장 완료
저장 위치: /content/drive/MyDrive/NoonGil/v12_no_curb_finetune_v10_best.pt

=== 최근 10 epoch 결과 ===
    epoch  metrics/mAP50(B)  metrics/mAP50-95(B)
40     41           0.88426              0.53120
41     42           0.88997              0.52261
42     43           0.88355              0.53256
43     44           0.87946              0.52938
44     45           0.86798              0.51042
45     46           0.87797              0.52533
46     47           0.86968              0.52337
47     48           0.87562              0.52847
48     49           0.87862              0.53233
49     50           0.87960              0.52732


## 클래스별 성능 비교


In [ ]:
# v6 기준 취약 클래스 성능 비교
print('=== v6 (기존) 취약 클래스 mAP50 ===')
v6_results = {
    'motorcycle':     0.615,
    'pavement_damage': 0.749,
    'ramp':           0.619,
    'step':           0.387,
}
for cls, val in v6_results.items():
    print(f'  {cls}: {val:.3f}')

print('\n=== v10 (fine-tuning 후) 결과는 위 학습 완료 후 확인 ===')
print('yolo detect val 명령어로 per-class mAP 확인:')
print(f'  !yolo detect val data=/content/datasets/merged/data.yaml model={DRIVE_BASE}/v12_no_curb_finetune_v10_best.pt')

=== v6 (기존) 취약 클래스 mAP50 ===
  motorcycle: 0.615
  pavement_damage: 0.749
  ramp: 0.619
  step: 0.387

=== v10 (fine-tuning 후) 결과는 위 학습 완료 후 확인 ===
yolo detect val 명령어로 per-class mAP 확인:
  !yolo detect val data=/content/datasets/merged/data.yaml model=/content/drive/MyDrive/NoonGil/v12_no_curb_finetune_v10_best.pt


## new version 검증 (학습 완료 후 실행)

In [ ]:
import os
os.chdir('/content/yolov12')

!yolo detect val \
  data="/content/datasets/merged/data.yaml" \
  model={DRIVE_BASE}/v12_no_curb_finetune_v10_best.pt \
  imgsz=640

FlashAttention is not available on this device. Using scaled_dot_product_attention instead.
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv12s summary (fused): 376 layers, 9,079,626 parameters, 0 gradients, 19.3 GFLOPs
val: Scanning /content/datasets/merged/valid/labels.cache... 148 images, 30 backgrounds, 0 corrupt: 100% 178/178 [00:00<?, ?it/s]
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 44, len(boxes) = 292. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% 12/12 [00:04<00:00,  2.66it/s]
                   all        178        292      0.872       0.84      0.883      0.544
               bicycle          8          8      0.958      0.875      0.911      0.599
               bollard          7  

In [ ]:
from pathlib import Path
import shutil

src = Path("/content/drive/MyDrive/NoonGil/v12_no_curb_finetune_v10_best.pt")
dst = Path("/content/drive/MyDrive/NoonGil/weights/v12_no_curb_finetune_v10_best.pt")

dst.parent.mkdir(parents=True, exist_ok=True)

shutil.copy2(src, dst)

print(f"✅ 원본: {src}")
print(f"✅ 백업 저장 완료: {dst}")

✅ 원본: /content/drive/MyDrive/NoonGil/v12_no_curb_finetune_v10_best.pt
✅ 백업 저장 완료: /content/drive/MyDrive/NoonGil/weights/v12_no_curb_finetune_v10_best.pt
